In [0]:
dbutils.widgets.text("catalogo", "mlops", "Catálogo")
dbutils.widgets.text("esquema", "frecuencia", "Esquema")
dbutils.widgets.text("modelo", "frecuencia_rc", "Modelo")
dbutils.widgets.text("alias", "champion", "Alias")
dbutils.widgets.text("lote", "1", "Lote")
dbutils.widgets.text("n_lotes", "4", "Número de lotes")
dbutils.widgets.text("inicio_nuevas", "80", "Primer balde de nuevas")
dbutils.widgets.text("deriva_bonusmalus", "0", "Deriva BonusMalus (simulación)")


catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
modelo_nombre = dbutils.widgets.get("modelo")
alias = dbutils.widgets.get("alias")
lote = int(dbutils.widgets.get("lote"))
n_lotes = int(dbutils.widgets.get("n_lotes"))
inicio_nuevas = int(dbutils.widgets.get("inicio_nuevas"))
deriva_bonusmalus = int(dbutils.widgets.get("deriva_bonusmalus"))

In [0]:
import mlflow
from mlflow import MlflowClient
from pyspark.sql import functions as F

nombre_modelo = f"{catalogo}.{esquema}.{modelo_nombre}"
tabla_nuevas = f"{catalogo}.{esquema}.polizas_nuevas"
tabla_pred = f"{catalogo}.{esquema}.predicciones"

version_modelo = int(MlflowClient().get_model_version_by_alias(nombre_modelo, alias).version)
modelo = mlflow.pyfunc.load_model(f"models:/{nombre_modelo}/{version_modelo}")
columnas = modelo.metadata.get_input_schema().input_names()

tamano_lote = (100 - inicio_nuevas) // n_lotes
datos = (
    spark.table(tabla_nuevas)
    .withColumn("lote", (F.floor((F.pmod(F.xxhash64("IDpol"), F.lit(100)) - inicio_nuevas) / tamano_lote) + 1).cast("int"))
    .filter(F.col("lote") == lote)
    .toPandas()
)
print("modelo v", version_modelo, "| lote", lote, "| pólizas", len(datos))

In [0]:
   #spark.sql(f"DROP TABLE IF EXISTS {tabla_pred}")

In [0]:
datos["BonusMalus"] = datos["BonusMalus"] + deriva_bonusmalus
datos["frecuencia_predicha"] = modelo.predict(datos[columnas]).astype("float64")
datos["siniestros_esperados"] = datos["frecuencia_predicha"] * datos["Exposure"]

pred = spark.createDataFrame(
    datos[["IDpol", "lote", "Exposure", *columnas, "frecuencia_predicha", "siniestros_esperados"]]
).withColumns({
    "modelo": F.lit(nombre_modelo),
    "version_modelo": F.lit(version_modelo),
    "deriva_bonusmalus": F.lit(deriva_bonusmalus),
    "fecha_scoring": F.current_timestamp(),
})

if spark.catalog.tableExists(tabla_pred):
    spark.sql(f"DELETE FROM {tabla_pred} WHERE lote = {lote}")
pred.write.mode("append").saveAsTable(tabla_pred)

In [0]:
display(spark.sql(f"""
    SELECT lote, version_modelo, COUNT(*) AS polizas, SUM(Exposure) AS exposicion,
           SUM(siniestros_esperados) AS esperados,
           SUM(siniestros_esperados) / SUM(Exposure) AS frecuencia_media
    FROM {tabla_pred}
    GROUP BY lote, version_modelo
    ORDER BY lote
"""))